<br>
<font>
<div dir=ltr align=center>
<div dir=ltr align=center>
<font color=0F5298 size=7>
    Artificial Intelligence <br>
<font color=2565AE size=5>
    Computer Engineering Department <br>
    Arash Marioriyad<br>
    Spring 2026<br>
<font color=3C99D size=5>
    Practical HomeWork 2<br>
    Nonogram<br>
<font color=696880 size=4>
    Foad Kheirabady

`Full Name:`

`Student ID:`

# Introduction

In this notebook, you will implement several algorithms for solving **Constraint Satisfaction Problems (CSPs)**.
We begin with a **baseline brute-force solver** that assigns values blindly and uses **backtracking** when a constraint is violated.
You will then progressively enhance it with:

- **Forward Checking** – pruning inconsistent domain values after each assignment,
- **Minimum Remaining Values (MRV)** – always choosing the most constrained variable next,
- **Arc Consistency (AC-3)** – enforcing global consistency across all variable domains.

## What is a CSP?

A **Constraint Satisfaction Problem** is defined by three components:

- **Variables:** the unknowns we need to assign values to.
- **Domains:** the set of possible values each variable can take.
- **Constraints:** rules restricting which combinations of values are valid.

## Nonogram as a CSP

**Nonograms** (also known as Picross or Griddlers) are grid-based logic puzzles.
Each cell in an $N \times M$ grid must be filled ($1$) or left empty ($0$).
The valid pattern is determined by **clues** given for each row and column.

A clue is a sequence of integers indicating the lengths of consecutive filled-cell blocks, in left-to-right
(or top-to-bottom) order, with **at least one empty cell** between consecutive blocks.

**Example:** the clue `[2, 1]` for a row of length 5 means: a block of 2 filled cells, at least one gap, then a block of 1 filled cell.
Valid arrangements include `11010`, `11001`, `01101`.

We model this as a CSP:

- **Variables:** every cell $(i, j)$ in the $N \times M$ grid.
- **Domain:** $\{0, 1\}$ for each cell ($0$ = empty, $1$ = filled).
- **Constraints:** for each row $i$, the sequence of filled cells must match `row_clues[i]`;
  for each column $j$, it must match `col_clues[j]`.
- **Neighbors** of $(i, j)$: all other cells sharing the same row or column.

An example Nonogram puzzle is shown below.

![Example Nonogram](Nonogram-of-a-Dolphin.webp)

## Helper Functions in `utils.py`

Two utility functions are provided and are **free to use** throughout this notebook:

| Function | Description |
|---|---|
| `get_valid_arrangements(clue, length)` | Returns **all** valid binary lists of the given length satisfying the clue. |
| `is_line_possible(values, clue)` | Given a partial line (list with $-1$ for unassigned), returns `True` if the clue can still be satisfied. |

Study these helpers before starting — they are the key building blocks for every solver.


## Grading
| Chapter | Points |
|---|---|
| Baseline Solver | 10 |
| Forward Checking | 30 |
| MRV | 10 |
| AC-3 | 50 |

**NOTE 1:** All value assignments **must** use the **`state.assign`** function.
Direct modification of the grid outside this function may result in loss of the chapter's grade.

**NOTE 2:** Runtime optimization is encouraged but not required — the key metric is the **number of assignments**.

**NOTE 3:** You earn chapter points only if your solver makes **fewer assignments** than the previous solver.

**NOTE 4:** Two examples are used throughout this notebook to evaluate each solver:
- **First Example (15×15)** — a moderately complex puzzle used for **testing and debugging**.
  It is intentionally designed to take 5–10 seconds on the baseline solver, giving a clear
  signal that each successive solver is making meaningful improvements in assignments and backtracks.
- **Second Example (15×20)** — the **main benchmark puzzle**. This is the primary puzzle
  on which solvers are graded. It is significantly harder and may take a long time on weaker solvers.
  
> ⚠️ **Important:** Each solver must show a measurable reduction in the number of assignments
> on **both** examples compared to the previous solver. Pay close attention to the second example
> — it is the one that determines your grade.

In [ ]:
import time
from collections import deque

import numpy as np

from models import NonogramCSP, Cell, StepLogger
from utils import (
    build_nonogram_csp, make_animation,
    get_valid_arrangements, is_line_possible,
    ROW_CLUES_1, COL_CLUES_1,
    ROW_CLUES_2, COL_CLUES_2
)


# Baseline Solver — 10 Points

In this chapter, implement the **baseline backtracking solver**.

Strategy:
- Iterate through cells in row-major order (left to right, top to bottom).
- For each unassigned cell, try each value in $\{0, 1\}$.
- If the value violates any constraint, skip it.
- If neither value works, **backtrack** to the previous assignment.

Do **not** use forward checking, MRV, or any domain pruning at this stage.

**Consistency check:** after tentatively assigning a value, call `is_line_possible`
on the current row and the current column. If either fails, the value is invalid.

## The `ValuesState` Class

`ValuesState` tracks the current assignment.  The grid is a NumPy array initialised
to $-1$ (unassigned). Always use **`state.assign`** to update a cell — it handles
the step logger so the animation remains consistent.

In [ ]:
class ValuesState:
    assignment: np.ndarray   # shape (N, M), dtype int,  -1 = unassigned

    def __init__(self, csp: NonogramCSP):
        self.assignment = np.full((csp.N, csp.M), -1, dtype=int)

    def assign(self, cell: Cell, v: int, csp: NonogramCSP, logger: StepLogger):
        """
        Assign value v to cell.
        Pass v = -1 to undo an assignment (backtrack).
        """
        if v == -1:
            logger.log("backtrack", cell, int(self.assignment[cell]))
        else:
            logger.log("assign", cell, v)
        self.assignment[cell] = v

In [ ]:
def solve_baseline(csp: NonogramCSP, state: ValuesState, logger: StepLogger) -> bool:
    """
    Solve the Nonogram using plain backtracking with no domain pruning.

    Assign cells in the order given by csp.variables (row-major).
    For each cell, try values [0, 1] and check consistency using
    is_line_possible on the affected row and column.
    Backtrack when neither value is consistent.

    Returns True if a solution is found, False otherwise.
    """

    def is_consistent(cell: Cell, value: int) -> bool:
        """
        Return True if assigning `value` to `cell` is consistent with
        the current partial assignment (check only this cell's row and column).

        Hint: temporarily update state.assignment[cell] before calling
        is_line_possible, then restore it. Do NOT call state.assign here.
        """
        # YOUR CODE HERE
        pass

    def backtrack() -> bool:
        # YOUR CODE HERE
        pass

    return backtrack()

In [ ]:
# Set to True to also run the second (harder) 15×20 example.
# Warning: it may take significantly longer to solve.
RUN_SECOND_EXAMPLE = True

def run_example(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = ValuesState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_baseline(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    # Return csp + logger so the animation section can use them
    return csp, logger


csp1, logger1 = run_example("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2, logger2 = None, None
if RUN_SECOND_EXAMPLE:
    csp2, logger2 = run_example("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)


In [ ]:
make_animation(csp1, logger1, fps=5, last_k=150, stride=1)

if csp2 is not None:
    make_animation(csp2, logger2, fps=5, last_k=150, stride=1)

# Forward Checking — 30 Points

In this chapter, extend the baseline solver with **forward checking**.

After each assignment to cell $(r, c)$, inspect **row $r$** and **column $c$**:

1. Compute all valid arrangements of that line consistent with the current assignment
   (use `get_valid_arrangements`, then filter by the current partial state).
2. For each **unassigned** cell in the line, determine which values still appear
   in at least one valid arrangement.
3. **Prune** values that never appear.
4. If any unassigned cell's domain becomes **empty**, the current branch fails — backtrack immediately.

Do **not** use the MRV heuristic yet; keep assigning cells in row-major order.

## The `DomainValuesState` Class

`DomainValuesState` extends `ValuesState` with per-cell `domains`.  
Always use **`state.prune`** and **`state.unprune`** to change domains — never
modify `domains` directly. These methods also update the step logger.

In [ ]:
class DomainValuesState(ValuesState):
    domains: dict   # Cell -> set of remaining values  {0, 1}

    def __init__(self, csp: NonogramCSP):
        super().__init__(csp)
        self.domains = {cell: {0, 1} for cell in csp.variables}

    def prune(self, cell: Cell, v: int, logger: StepLogger):
        self.domains[cell].discard(v)
        logger.log("prune", cell, v)

    def unprune(self, cell: Cell, v: int):
        self.domains[cell].add(v)

In [ ]:
def solve_forward_check(csp: NonogramCSP, state: DomainValuesState, logger: StepLogger) -> bool:
    """
    Solve using backtracking + one-step forward checking.

    After assigning a cell (r, c), call forward_check(r, c) which:
      - Checks row r and column c.
      - Prunes values from unassigned cells in those lines that are not
        supported by any valid arrangement.
      - Returns the list of (cell, value) pairs pruned, or None on failure.

    On backtracking, restore all pruned values with state.unprune.

    Returns True if solved, False otherwise.
    """

    def check_line(cells: list, clue: tuple, length: int):
        """
        Forward-check one line.
        Returns list of (cell, value) pruned, or None if a domain became empty.

        Steps:
          1. Build the list of current values for all cells in the line
             (use int(state.assignment[cell]), which is -1 for unassigned).
          2. Filter get_valid_arrangements(clue, length) to keep only arrangements
             consistent with current values.
          3. If no valid arrangement remains, return None.
          4. For each unassigned cell, prune values absent from all valid arrangements.
          5. If any domain becomes empty after pruning, return None.
        """
        # YOUR CODE HERE
        pass

    def forward_check(r: int, c: int):
        """
        Run check_line on row r, then on column c.
        If either fails, undo any pruning done so far and return None.
        Otherwise return the combined pruned list.
        """
        # YOUR CODE HERE
        pass

    def backtrack() -> bool:
        # YOUR CODE HERE
        pass

    return backtrack()

In [ ]:
def run_example_fc(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = DomainValuesState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_forward_check(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    return csp, logger


csp1_fc, logger1_fc = run_example_fc("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2_fc, logger2_fc = None, None
if RUN_SECOND_EXAMPLE:
    csp2_fc, logger2_fc = run_example_fc("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)



In [ ]:
make_animation(csp1_fc, logger1_fc, fps=5, last_k=150, stride=1)

if csp2_fc is not None:
    make_animation(csp2_fc, logger2_fc, fps=5, last_k=150, stride=1)

Discuss your results. In what ways did forward checking improve over the baseline? Is it always faster?

`Your Answer:`

# Minimum Remaining Values (MRV) — 10 Points

In this chapter, add the **MRV heuristic** on top of forward checking.

Instead of assigning cells in fixed row-major order, always pick the **unassigned cell
with the fewest remaining domain values**.  Targeting the most constrained cell first
tends to expose failures earlier, pruning large branches of the search tree.

Continue using **one-step forward checking** (exactly as in Chapter 2) after each assignment.

## The `MRVState` Class

`MRVState` extends `DomainValuesState` with a `num_remaining_values` array for fast
MRV lookups.  Pruning and unpruning automatically keep this count consistent.

In [ ]:
class MRVState(DomainValuesState):
    num_remaining_values: np.ndarray   # shape (N, M)

    def __init__(self, csp: NonogramCSP):
        super().__init__(csp)
        self.num_remaining_values = np.full((csp.N, csp.M), 2, dtype=int)

    def prune(self, cell: Cell, v: int, logger: StepLogger):
        self.domains[cell].discard(v)
        self.num_remaining_values[cell] -= 1
        logger.log("prune", cell, v)

    def unprune(self, cell: Cell, v: int):
        self.domains[cell].add(v)
        self.num_remaining_values[cell] += 1

In [ ]:
def solve_mrv(csp: NonogramCSP, state: MRVState, logger: StepLogger) -> bool:
    """
    Solve using backtracking + forward checking + MRV variable ordering.

    Replace the fixed-order cell selection with select_unassigned(), which
    returns the unassigned cell with the smallest domain
    (use state.num_remaining_values for efficiency).

    The forward_check logic is identical to Chapter 2.

    Returns True if solved, False otherwise.
    """

    def select_unassigned() -> Cell:
        """Return the unassigned cell with the fewest remaining values, or None if all cells are assigned."""
        # YOUR CODE HERE
        pass

    def check_line(cells: list, clue: tuple, length: int):
        """Same as Chapter 2."""
        # YOUR CODE HERE
        pass

    def forward_check(r: int, c: int):
        """Same as Chapter 2."""
        # YOUR CODE HERE
        pass

    def backtrack() -> bool:
        # YOUR CODE HERE
        pass

    return backtrack()

In [ ]:
def run_example_mrv(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = MRVState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_mrv(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    return csp, logger


csp1_mrv, logger1_mrv = run_example_mrv("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2_mrv, logger2_mrv = None, None
if RUN_SECOND_EXAMPLE:
    csp2_mrv, logger2_mrv = run_example_mrv("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)


In [ ]:
make_animation(csp1_mrv, logger1_mrv, fps=5, last_k=150, stride=1)

if csp2_mrv is not None:
    make_animation(csp2_mrv, logger2_mrv, fps=5, last_k=150, stride=1)

Compare your results to the previous solvers. How did MRV affect assignments and backtracks? Why does choosing the most constrained cell first help?

`Your Answer:`

# Arc Consistency (AC-3) — 50 Points

In this chapter, integrate **AC-3** into the MRV solver.

### Why AC-3?

Forward checking only re-examines the **two lines** directly touched by the most recent
assignment.  AC-3 goes further: whenever a pruning changes a cell's domain, it re-queues
**all other lines containing that cell**, propagating the constraint like a cascade.
This often eliminates many values (or forces assignments) long before backtracking would
reach those cells.

### AC-3 for Nonogram

An **arc** here corresponds to a **line** — a complete row or column.  The AC-3 queue holds
`('row', i)` and `('col', j)` items.

**Algorithm:**

1. Seed the queue (all lines at startup; just the affected row and column after an assignment).
2. Dequeue a line and compute all valid arrangements consistent with current assignments and domains.
3. For each unassigned cell in the line, prune values absent from every valid arrangement.
4. If a cell's domain **changed**, add its **other-dimension** line back to the queue
   (`'col'` for a cell in a row, and `'row'` for a cell in a column).
5. If any domain becomes empty, return failure.

Continue using **MRV** for cell selection.

**Implementation note:** `ac3` should return `(success: bool, pruned: list)`.
On backtracking, restore all items in `pruned` with `state.unprune`.

**Hint:** call `ac3()` once before the first assignment to reduce domains globally.

In [ ]:
def solve_ac3_mrv(csp: NonogramCSP, state: MRVState, logger: StepLogger) -> bool:
    """
    Solve using backtracking + AC-3 propagation + MRV variable ordering.

    ac3(initial_queue):
      - If initial_queue is None, seed with all rows and columns.
      - Otherwise seed with the provided list of ('row'/'col', idx) items.
      - Propagate until the queue is empty or a domain becomes empty.
      - Return (success, all_pruned).

    On backtrack: restore all pruned values, then undo the assignment.

    Returns True if solved, False otherwise.
    """

    def ac3(initial_queue=None):
        """
        Run AC-3.
        Returns (success: bool, all_pruned: list[tuple[Cell, int]]).
        """
        # YOUR CODE HERE
        pass

    def select_unassigned() -> Cell:
        """MRV cell selection — same as Chapter 3."""
        # YOUR CODE HERE
        pass

    def backtrack() -> bool:
        # YOUR CODE HERE
        pass

    # Run AC-3 before the first assignment
    success, _ = ac3()
    if not success:
        return False

    return backtrack()

In [ ]:
def run_example_ac3(label, row_clues, col_clues, timeout=None):
    import threading

    csp    = build_nonogram_csp(row_clues, col_clues)
    logger = StepLogger()
    state  = MRVState(csp)

    result = {"ok": None, "runtime": None, "timed_out": False}

    def _solve():
        t0 = time.time()
        result["ok"]      = solve_ac3_mrv(csp, state, logger)
        result["runtime"] = time.time() - t0

    thread = threading.Thread(target=_solve, daemon=True)
    thread.start()
    thread.join(timeout)

    if thread.is_alive():
        result["timed_out"] = True
        result["runtime"]   = timeout

    num_assigns    = sum(1 for e in logger.events if e.action == "assign")
    num_prunes     = sum(1 for e in logger.events if e.action == "prune")
    num_backtracks = sum(1 for e in logger.events if e.action == "backtrack")

    print(f"\n{label}")
    if result["timed_out"]:
        print(f"Solved:      TIMED OUT after {timeout}s")
    else:
        print(f"Solved:      {result['ok']}")
    print(f"Runtime:     {result['runtime']:.3f} seconds")
    print(f"Assignments: {num_assigns}")
    print(f"Prunes:      {num_prunes}")
    print(f"Backtracks:  {num_backtracks}")

    return csp, logger


csp1_ac3, logger1_ac3 = run_example_ac3("First Example (15×15)", ROW_CLUES_1, COL_CLUES_1)

csp2_ac3, logger2_ac3 = None, None
if RUN_SECOND_EXAMPLE:
    csp2_ac3, logger2_ac3 = run_example_ac3("Second Example (15×20)", ROW_CLUES_2, COL_CLUES_2)


In [ ]:
make_animation(csp1_ac3, logger1_ac3, fps=5, last_k=150, stride=1)

if csp2_ac3 is not None:
    make_animation(csp2_ac3, logger2_ac3, fps=5, last_k=150, stride=1)

Discuss your results. Did AC-3 improve runtime compared to MRV alone? Is AC-3 always beneficial? What are its tradeoffs?

`Your Answer:`